In [6]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

Load  cleaned data

In [7]:
df = pd.read_csv('../data/processed/filtered_complaints.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(1343, 20)
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'product_category', 'cleaned_narrative']


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,product_category,cleaned_narrative
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121.0,Credit Card,a card was opened under my name by a fraudster...
1,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,NaN,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085.0,Credit Card,dear cfpb i have a secured credit card with ci...
2,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,NaN,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217.0,Credit Card,i have a citi rewards cards the credit balance...
3,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411.0,Credit Card,i am writing to dispute the following charges ...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem when making payments,Problem during payment process,"Although the account had been deemed closed, I...",Company believes it acted appropriately as aut...,Atlanticus Services Corporation,NY,11212,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13965746.0,Credit Card,although the account had been deemed closed i ...


In [9]:
# Fix: some rows have NaN or empty cleaned_narrative after saving/reloading
df = df.dropna(subset=['cleaned_narrative'])
df = df[df['cleaned_narrative'].str.strip() != '']

print(f'Rows ready for chunking: {len(df)}')
print(df['Product'].value_counts().to_string())

Rows ready for chunking: 1342
Product
Credit card                                                893
Money transfer, virtual currency, or money service         201
Payday loan, title loan, personal loan, or advance loan    123
Checking or savings account                                123
Credit card or prepaid card                                  2


Chunk the narratives

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks, metadatas, ids = [], [], []

for idx, row in df.iterrows():
    pieces = splitter.split_text(row['cleaned_narrative'])
    for i, piece in enumerate(pieces):
        chunks.append(piece)
        metadatas.append({
            'complaint_id': str(row['Complaint ID']),   # adjust column name if different
            'product_category': row['product_category'],
            'chunk_index': i,
            'total_chunks': len(pieces)
        })
        ids.append(f"{idx}_{i}")

print(f"{len(df)} complaints → {len(chunks)} chunks")

1342 complaints → 3910 chunks


In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks, show_progress_bar=True, batch_size=32)

print(f'Embeddings shape: {embeddings.shape}')
print(f'Expected: ({len(chunks)}, 384)')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Envy\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Envy\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/123 [00:01<?, ?it/s]

Embeddings shape: (3910, 384)
Expected: (3910, 384)


In [12]:

import os

os.makedirs('../vector_store', exist_ok=True)

client = chromadb.PersistentClient(path='../vector_store')

# Delete collection if it exists from a previous run
try:
    client.delete_collection(name='complaint_chunks')
except:
    pass

collection = client.create_collection(name='complaint_chunks')

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas
)

print(f'Vector store created successfully')
print(f'Total chunks stored: {collection.count()}')

Vector store created successfully
Total chunks stored: 3910


In [13]:
query = "Why are people unhappy with credit cards?"
query_embedding = model.encode([query]).tolist()

results = collection.query(query_embeddings=query_embedding, n_results=3)

for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"Result {i+1}")
    print(f"Product: {meta['product_category']}")
    print(f"Complaint ID: {meta['complaint_id']}")
    print(f"Text: {doc[:200]}")
    print("---")

Result 1
Product: Credit Card
Complaint ID: 13891123.0
Text: card companies and banks have enormous power and consumers are left with situations like this they should be required to let you know at the point of sale what the problem is
---
Result 2
Product: Credit Card
Complaint ID: 13706123.0
Text: i have good credit when a loyal long term customer of citibank tries to negotiate a lower interest rate and receives no assistance whatsoever that customer may decide to declare bankruptcy maybe that 
---
Result 3
Product: Credit Card
Complaint ID: 12992871.0
Text: americans which i can hardly blame them in this government environment but again no normal credit card companies close accounts like this without notice
---
